The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

# 循环神经网络的简洁实现
:label:`sec_rnn-concise`

虽然 :numref:`sec_rnn_scratch`
对了解循环神经网络的实现方式具有指导意义，但并不方便。
本节将展示如何使用深度学习框架的高级API提供的函数更有效地实现相同的语言模型。
我们仍然从读取时光机器数据集开始。


In [2]:
import re
import collections
import requests
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

class Vocab:
    """字符级词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []

        counter = collections.Counter(tokens)

        self.token_freqs = sorted(
            counter.items(),
            key=lambda x: x[1],
            reverse=True
        )

        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {
            token: idx for idx, token in enumerate(self.idx_to_token)
        }

        for token, freq in self.token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if isinstance(tokens, (list, tuple)):
            return [self.__getitem__(token) for token in tokens]
        return self.token_to_idx.get(tokens, self.token_to_idx['<unk>'])

    def to_tokens(self, indices):
        if isinstance(indices, (list, tuple)):
            return [self.idx_to_token[int(i)] for i in indices]
        return self.idx_to_token[int(indices)]


def read_time_machine():
    """读取 Time Machine 数据集"""
    url = "https://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt"
    response = requests.get(url)
    response.raise_for_status()

    lines = response.text.splitlines()

    # 和 D2L 类似：只保留字母，其他字符替换为空格，并转小写
    return [
        re.sub('[^A-Za-z]+', ' ', line).strip().lower()
        for line in lines
    ]


def tokenize(lines, token='char'):
    """分词"""
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        raise ValueError("token must be 'word' or 'char'")


def load_corpus_time_machine(max_tokens=-1):
    """返回字符索引序列 corpus 和词表 vocab"""
    lines = read_time_machine()
    tokens = tokenize(lines, token='char')

    # Time Machine 默认是字符级语言模型
    corpus = [token for line in tokens for token in line]

    vocab = Vocab(corpus)

    corpus = vocab[corpus]

    if max_tokens > 0:
        corpus = corpus[:max_tokens]

    return corpus, vocab


def seq_data_iter_random(corpus, batch_size, num_steps):
    """使用随机采样生成一个小批量子序列"""
    # 从随机偏移量开始
    corpus = corpus[torch.randint(0, num_steps, (1,)).item():]

    # 减去 1，因为需要 label 是下一个 token
    num_subseqs = (len(corpus) - 1) // num_steps

    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    initial_indices = torch.tensor(initial_indices)
    initial_indices = initial_indices[torch.randperm(len(initial_indices))]

    def data(pos):
        return corpus[pos: pos + num_steps]

    num_batches = num_subseqs // batch_size

    for i in range(0, batch_size * num_batches, batch_size):
        batch_indices = initial_indices[i: i + batch_size]

        X = torch.tensor(
            [data(int(j)) for j in batch_indices],
            dtype=torch.long
        )

        Y = torch.tensor(
            [data(int(j) + 1) for j in batch_indices],
            dtype=torch.long
        )

        yield X, Y

class SeqDataLoader:
    """序列数据加载器"""
    def __init__(self, batch_size, num_steps, max_tokens=10000):
        self.corpus, self.vocab = load_corpus_time_machine(max_tokens)
        self.batch_size = batch_size
        self.num_steps = num_steps

    def __iter__(self):
        return seq_data_iter_random(
            self.corpus,
            self.batch_size,
            self.num_steps
        )

def load_data_time_machine(batch_size, num_steps, max_tokens=10000):
    """返回 Time Machine 数据迭代器和词表"""
    data_iter = SeqDataLoader(batch_size, num_steps, max_tokens)
    return data_iter, data_iter.vocab

In [3]:
import torch
from torch import nn
from torch.nn import functional as F

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

## [**定义模型**]

高级API提供了循环神经网络的实现。
我们构造一个具有256个隐藏单元的单隐藏层的循环神经网络层`rnn_layer`。
事实上，我们还没有讨论多层循环神经网络的意义（这将在 :numref:`sec_deep_rnn`中介绍）。
现在仅需要将多层理解为一层循环神经网络的输出被用作下一层循环神经网络的输入就足够了。


In [4]:
num_hiddens = 256
rnn_layer = nn.RNN(len(vocab), num_hiddens)

我们(**使用张量来初始化隐状态**)，它的形状是（隐藏层数，批量大小，隐藏单元数）。


In [5]:
state = torch.zeros((1, batch_size, num_hiddens))
state.shape

torch.Size([1, 32, 256])

[**通过一个隐状态和一个输入，我们就可以用更新后的隐状态计算输出。**]
需要强调的是，`rnn_layer`的“输出”（`Y`）不涉及输出层的计算：
它是指每个时间步的隐状态，这些隐状态可以用作后续输出层的输入。


In [6]:
X = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape

(torch.Size([35, 32, 256]), torch.Size([1, 32, 256]))

与 :numref:`sec_rnn_scratch`类似，
[**我们为一个完整的循环神经网络模型定义了一个`RNNModel`类**]。
注意，`rnn_layer`只包含隐藏的循环层，我们还需要创建一个单独的输出层。


In [8]:
class RNNModel(nn.Module):
    """循环神经网络模型"""
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        # 如果RNN是双向的（之后将介绍），num_directions应该是2，否则应该是1
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        Y, state = self.rnn(X, state)
        # 全连接层首先将Y的形状改为(时间步数*批量大小,隐藏单元数)
        # 它的输出形状是(时间步数*批量大小,词表大小)。
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device, batch_size=1):
        if not isinstance(self.rnn, nn.LSTM):
            # nn.GRU以张量作为隐状态
            return  torch.zeros((self.num_directions * self.rnn.num_layers,
                                 batch_size, self.num_hiddens),
                                device=device)
        else:
            # nn.LSTM以元组作为隐状态
            return (torch.zeros((
                self.num_directions * self.rnn.num_layers,
                batch_size, self.num_hiddens), device=device),
                    torch.zeros((
                        self.num_directions * self.rnn.num_layers,
                        batch_size, self.num_hiddens), device=device))

## 训练与预测

在训练模型之前，让我们[**基于一个具有随机权重的模型进行预测**]。


In [9]:
import torch
from torch.nn import functional as F


def try_gpu():
    """如果有 GPU，则使用 GPU，否则使用 CPU"""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def predict_ch8(prefix, num_preds, net, vocab, device):
    """在 prefix 后面生成新字符，不依赖 d2l"""
    net.eval()

    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]

    def get_input():
        return torch.tensor([[outputs[-1]]], device=device)

    # 预热：用 prefix 中已有字符更新 RNN 状态
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])

    # 逐字符预测
    for _ in range(num_preds):
        y_hat, state = net(get_input(), state)
        next_token = int(y_hat.argmax(dim=1).reshape(1))
        outputs.append(next_token)

    return ''.join(vocab.to_tokens(outputs))

In [10]:
device = try_gpu()
net = RNNModel(rnn_layer, vocab_size=len(vocab))
net = net.to(device)
predict_ch8('time traveller', 10, net, vocab, device)

'time travellerevev uviyp'

很明显，这种模型根本不能输出好的结果。
接下来，我们使用 :numref:`sec_rnn_scratch`中
定义的超参数调用`train_ch8`，并且[**使用高级API训练模型**]。


In [11]:
import math
import torch
from torch import nn

num_epochs, lr = 500, 1

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=lr)

for epoch in range(num_epochs):
    state = None
    total_loss = 0.0
    total_tokens = 0

    net.train()

    for X, Y in train_iter:
        X, Y = X.to(device), Y.to(device)

        # 初始化隐状态
        if state is None:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            # 截断梯度，避免反向传播穿过整个历史序列
            if isinstance(state, tuple):  # LSTM
                state = tuple(s.detach() for s in state)
            else:  # RNN / GRU
                state = state.detach()

        # Y shape: (batch_size, num_steps)
        # 转成一维标签: (batch_size * num_steps,)
        y = Y.T.reshape(-1)

        # y_hat shape: (batch_size * num_steps, vocab_size)
        y_hat, state = net(X, state)

        l = loss(y_hat, y.long())

        optimizer.zero_grad()
        l.backward()

        # 梯度裁剪
        params = [
            p for p in net.parameters()
            if p.requires_grad and p.grad is not None
        ]
        norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))

        if norm > 1:
            for param in params:
                param.grad[:] *= 1 / norm

        optimizer.step()

        total_loss += l.item() * y.numel()
        total_tokens += y.numel()

    ppl = math.exp(total_loss / total_tokens)

    if (epoch + 1) % 10 == 0:
        print(
            f"epoch {epoch + 1}, "
            f"perplexity {ppl:.1f}, "
            f"{predict_ch8('time traveller', 50, net, vocab, device)}"
        )

epoch 10, perplexity 11.2, time travellere the the the the the the the the the the the the 
epoch 20, perplexity 9.5, time traveller and and and and and and and and and and and and a
epoch 30, perplexity 8.8, time travellere the the the the the the the the the the the the 
epoch 40, perplexity 8.2, time traveller the that have the that ing the the that ing the t
epoch 50, perplexity 7.7, time travellere we the the the the the the the the the the the t
epoch 60, perplexity 7.5, time traveller and her all are ware wat ous an the the the the t
epoch 70, perplexity 7.2, time traveller and his this this the that he pore his this this 
epoch 80, perplexity 6.7, time traveller and he said the medint si d an ther and he the pa
epoch 90, perplexity 6.0, time travelleris that is ally io jurid the the the that is ally 
epoch 100, perplexity 5.6, time traveller abug the groug the rait to hall has soul and we h
epoch 110, perplexity 5.3, time traveller smiled another at and ho the than a spale on t

In [12]:
print(predict_ch8('time traveller', 50, net, vocab, device))
print(predict_ch8('traveller', 50, net, vocab, device))

time traveller smiled round at us then still smiling faintlyand 
traveller smiled round at us then still smiling faintlyand 


与上一节相比，由于深度学习框架的高级API对代码进行了更多的优化，
该模型在较短的时间内达到了较低的困惑度。

## 小结

* 深度学习框架的高级API提供了循环神经网络层的实现。
* 高级API的循环神经网络层返回一个输出和一个更新后的隐状态，我们还需要计算整个模型的输出层。
* 相比从零开始实现的循环神经网络，使用高级API实现可以加速训练。

## 练习

1. 尝试使用高级API，能使循环神经网络模型过拟合吗？
1. 如果在循环神经网络模型中增加隐藏层的数量会发生什么？能使模型正常工作吗？
1. 尝试使用循环神经网络实现 :numref:`sec_sequence`的自回归模型。


[Discussions](https://discuss.d2l.ai/t/2106)
